#   LangGraph 활용 - Corrective RAG (CRAG)

---

## 1. 환경 설정

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
import warnings
import logging
from datetime import datetime
import operator
from typing import TypedDict, Union, List, Dict, Tuple, Any, Annotated
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# 경고 무시 설정
warnings.filterwarnings("ignore")

# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## 2. Corrective RAG (CRAG) 구현
- CRAG (Corrective Retrieval-Augmented Generation) 
- 논문: https://arxiv.org/pdf/2401.15884

- 주요 과정: 검색 -> 평가 -> 지식 정제 또는 웹 검색 -> 답변 생성

   1. 문서 관련성 평가 (`grade_documents`):
      - 각 문서의 관련성을 평가
      - 기준을 통과하는 문서만을 유지

   1. 지식 정제 (`refine_knowledge`):
      - 문서를 "지식 조각"으로 분할하고 각각의 관련성을 평가
      - 관련성 높은(0.5 초과) 지식 조각만 유지

   1. 웹 검색 (`web_search`):
      - 문서가 충분한 정보를 담지 못한 경우 외부 지식을 활용
      - 웹 검색 결과를 기존 문서에 추가 

   1. 답변 생성 (`generate_answer`):
      - 정제된 지식 조각을 사용하여 답변을 생성
      - 관련 정보가 없을 경우 적절한 메시지를 반환



###  2-1. Tool 정의

`(1) 벡터저장소 검색기`

In [3]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

class AdaptiveVectorStore:
    """적응형 검색을 위한 벡터 저장소"""
    
    def __init__(self, collection_name: str, persist_dir: str = "./chroma_db"):
        # 임베딩 모델 사용 
        self.embeddings = OpenAIEmbeddings(
            model="text-embedding-3-small",
            dimensions=1536  # 차원 설정
        )
        
        # Chroma DB 초기화
        self.vector_db = Chroma(
            embedding_function=self.embeddings,
            collection_name=collection_name,
            persist_directory=persist_dir,
        )
        
        # 다단계 검색 전략
        self.search_configs = {
            "initial": {"k": 3, "score_threshold": 0.3},
            "expanded": {"k": 5, "score_threshold": 0.1},
            "exhaustive": {"k": 10, "score_threshold": 0.0}
        }

        logger.info(f"✅ Vector store '{collection_name}' initialized")

    def multi_stage_search(self, query: str):
        """단계별 검색 전략"""
        # 1차: 정밀 검색
        initial = self.vector_db.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs=self.search_configs["initial"]
        ).invoke(query)

        logger.info(f"🔍 1차 검색 모드: {len(initial)}개의 문서 검색")

        if len(initial) < 1:
            # 2차: 확장 검색
            expanded = self.vector_db.as_retriever(
                search_type="similarity_score_threshold",
                search_kwargs=self.search_configs["expanded"]
            ).invoke(query)

            logger.info(f"🔍 2차 검색 모드: {len(expanded)}개의 문서 검색")

            if len(expanded) < 1:
                # 3차: 포괄 검색
                exhaustive = self.vector_db.as_retriever(
                    search_type="similarity_score_threshold",
                    search_kwargs=self.search_configs["exhaustive"]
                ).invoke(query)

                logger.info(f"🔍 3차 검색 모드: {len(exhaustive)}개의 문서 검색")

                return exhaustive
            
            return expanded
        else:
            return initial


# 사용 예시
vector_db = AdaptiveVectorStore(collection_name="restaurant_menu")
results = vector_db.multi_stage_search("채식주의자를 위한 메뉴가 있나요?")

2025-09-29 14:14:42,125 - chromadb.telemetry.product.posthog - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2025-09-29 14:14:42,255 - __main__ - INFO - ✅ Vector store 'restaurant_menu' initialized
2025-09-29 14:14:43,286 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:14:43,348 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.3
2025-09-29 14:14:43,349 - __main__ - INFO - 🔍 1차 검색 모드: 0개의 문서 검색
2025-09-29 14:14:44,148 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:14:44,151 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.1
2025-09-29 14:14:44,152 - __main__ - INFO - 🔍 2차 검색 모드: 0개의 문서 검색
2025-09-29 14:14:44,914 - httpx - INFO - HTTP Request: POST https://api.openai.c

In [4]:
for result in results:
    print(result)

page_content='1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.' metadata={'source': './data/restaurant_menu.txt', 'menu_name': '시그니처 스테이크', 'menu_number': 1}
page_content='1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.' metadata={'source': './data/restaurant_menu.txt', 'menu_name': '시그니처 스테이크', 'menu_number': 1}
page_content='1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.' metadata={'menu_name': '시그니처 스테이크', 'menu_number': 1, 'source': './data/restaurant_menu.txt'}
page

`(2) 웹 검색`

In [7]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(max_results=5)

search_tool.invoke("스테이크와 어울리는 와인")

ValidationError: 1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

### 2-2. LLM 모델

`(1) Retrieval Grader`

In [8]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 문서 관련성 평가 결과를 위한 데이터 모델 정의
class GradeDocuments(BaseModel):
    """Three-class score for relevance check on retrieved documents."""
    relevance_score: Literal["correct", "incorrect", "ambiguous"] = Field(
        description="Document relevance to the question: 'correct', 'incorrect', or 'ambiguous'"
    )

# LLM 모델 초기화 및 구조화된 출력 설정
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# 문서 관련성 평가를 위한 시스템 프롬프트 정의
system_prompt = """
You are an expert evaluator tasked with assessing the relevance of retrieved documents to a user's question. Your role is crucial in enhancing the quality of information retrieval systems.

[평가 기준]
1. 키워드 관련성: 문서가 질문의 주요 단어나 유사어를 포함하는지 확인
2. 의미적 관련성: 문서의 전반적인 주제가 질문의 의도와 일치하는지 평가
3. 부분 관련성: 질문의 일부를 다루거나 맥락 정보를 제공하는 문서도 고려
4. 답변 가능성: 직접적인 답이 아니더라도 답변 형성에 도움될 정보 포함 여부 평가

[점수 체계]
- 'correct': 문서가 명확히 관련 있고, 질문에 답하는 데 필요한 정보를 포함함.
- 'incorrect': 문서가 명확히 무관하거나, 질문에 도움이 되지 않는 정보를 포함함.
- 'ambiguous': 문서의 관련성이 불분명하거나, 일부 관련 정보는 있지만 유용성이 확실하지 않음, 혹은 질문과 약간만 관련 있음.

[주의사항]
- 단순 단어 매칭이 아닌 질문의 전체 맥락을 고려하세요
- 완벽한 답변이 아니어도 유용한 정보가 있다면 관련 있다고 판단하세요

Your evaluation plays a critical role in improving the overall performance of the information retrieval system. Strive for balanced and thoughtful assessments.
"""

# 채점 프롬프트 템플릿 생성
grade_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Document: \n\n {document} \n\n Question: {question}"),
])

# Retrieval Grader 파이프라인 구성
retrieval_grader = grade_prompt | structured_llm_grader
    
# 관련성 평가 실행
question = "채식주의자를 위한 메뉴가 있나요?"
retrieved_docs = vector_db.multi_stage_search(question)
print(f"검색된 문서 수: {len(retrieved_docs)}")

for test_chunk in retrieved_docs:
    print("문서:", test_chunk.page_content)

    relevance = retrieval_grader.invoke({"question": question, "document": test_chunk.page_content})
    print(f"문서 관련성: {relevance.relevance_score}")
    print("=====================================")

2025-09-29 14:22:21,808 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:21,824 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.3
2025-09-29 14:22:21,826 - __main__ - INFO - 🔍 1차 검색 모드: 0개의 문서 검색
2025-09-29 14:22:22,531 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:22,533 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.1
2025-09-29 14:22:22,534 - __main__ - INFO - 🔍 2차 검색 모드: 0개의 문서 검색
2025-09-29 14:22:23,331 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:23,335 - __main__ - INFO - 🔍 3차 검색 모드: 10개의 문서 검색


검색된 문서 수: 10
문서: 1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.


2025-09-29 14:22:24,157 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


문서 관련성: incorrect
문서: 1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.


2025-09-29 14:22:24,751 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

문서 관련성: incorrect
문서: 3. 연어 타르타르
   • 가격: ₩18,000
   • 주요 식재료: 노르웨이산 생연어, 아보카도, 케이퍼, 적양파
   • 설명: 신선한 노르웨이산 생연어를 곱게 다져 아보카도, 케이퍼, 적양파와 함께 섞어 만든 타르타르입니다. 레몬 드레싱으로 상큼한 맛을 더했으며, 바삭한 브리오쉬 토스트와 함께 제공됩니다. 전채요리로 완벽한 메뉴입니다.


2025-09-29 14:22:28,715 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


문서 관련성: incorrect
문서: 4. 버섯 크림 수프
   • 가격: ₩10,000
   • 주요 식재료: 양송이버섯, 표고버섯, 생크림, 트러플 오일
   • 설명: 양송이버섯과 표고버섯을 오랜 시간 정성스레 끓여 만든 크림 수프입니다. 부드러운 텍스처와 깊은 버섯 향이 특징이며, 최상급 트러플 오일을 살짝 뿌려 고급스러운 향을 더했습니다. 파슬리를 곱게 다져 고명으로 올려 제공됩니다.


2025-09-29 14:22:29,407 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


문서 관련성: ambiguous


In [9]:
question = "해산물 요리를 추천해주세요."
retrieved_docs = vector_db.multi_stage_search(question)
print(f"검색된 문서 수: {len(retrieved_docs)}")

for test_chunk in retrieved_docs:
    print("문서:", test_chunk.page_content)

    relevance = retrieval_grader.invoke({"question": question, "document": test_chunk.page_content})
    print(f"문서 관련성: {relevance.relevance_score}")
    print("=====================================")

2025-09-29 14:22:32,827 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:32,831 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.3
2025-09-29 14:22:32,832 - __main__ - INFO - 🔍 1차 검색 모드: 0개의 문서 검색
2025-09-29 14:22:33,102 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:33,104 - __main__ - INFO - 🔍 2차 검색 모드: 5개의 문서 검색


검색된 문서 수: 5
문서: 6. 해산물 파스타
   • 가격: ₩24,000
   • 주요 식재료: 링귀네 파스타, 새우, 홍합, 오징어, 토마토 소스
   • 설명: 알 덴테로 삶은 링귀네 파스타에 신선한 해산물을 듬뿍 올린 메뉴입니다. 토마토 소스의 산미와 해산물의 감칠맛이 조화를 이루며, 마늘과 올리브 오일로 풍미를 더했습니다. 파슬리를 뿌려 향긋한 맛을 더합니다.


2025-09-29 14:22:34,034 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


문서 관련성: correct
문서: 6. 해산물 파스타
   • 가격: ₩24,000
   • 주요 식재료: 링귀네 파스타, 새우, 홍합, 오징어, 토마토 소스
   • 설명: 알 덴테로 삶은 링귀네 파스타에 신선한 해산물을 듬뿍 올린 메뉴입니다. 토마토 소스의 산미와 해산물의 감칠맛이 조화를 이루며, 마늘과 올리브 오일로 풍미를 더했습니다. 파슬리를 뿌려 향긋한 맛을 더합니다.


2025-09-29 14:22:34,578 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

문서 관련성: correct
문서: 30. 씨푸드 빠에야
    • 가격: ₩42,000
    • 주요 식재료: 스페인산 봄바 쌀, 홍합, 새우, 오징어, 사프란
    • 설명: 스페인 전통 방식으로 조리한 해산물 빠에야입니다. 최상급 사프란으로 노란빛을 내고 신선한 해산물을 듬뿍 넣어 지중해의 맛을 그대로 담았습니다. 2인 이상 주문 가능하며 레몬을 곁들여 제공됩니다.


2025-09-29 14:22:35,951 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


문서 관련성: correct
문서: 30. 씨푸드 빠에야
    • 가격: ₩42,000
    • 주요 식재료: 스페인산 봄바 쌀, 홍합, 새우, 오징어, 사프란
    • 설명: 스페인 전통 방식으로 조리한 해산물 빠에야입니다. 최상급 사프란으로 노란빛을 내고 신선한 해산물을 듬뿍 넣어 지중해의 맛을 그대로 담았습니다. 2인 이상 주문 가능하며 레몬을 곁들여 제공됩니다.


2025-09-29 14:22:36,530 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


문서 관련성: correct


In [10]:
question = "시그니처 메뉴가 있나요?"
retrieved_docs = vector_db.multi_stage_search(question)
print(f"검색된 문서 수: {len(retrieved_docs)}")

for test_chunk in retrieved_docs:
    print("문서:", test_chunk.page_content)

    relevance = retrieval_grader.invoke({"question": question, "document": test_chunk.page_content})
    print(f"문서 관련성: {relevance.relevance_score}")
    print("=====================================")

2025-09-29 14:22:39,239 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:39,243 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.3
2025-09-29 14:22:39,244 - __main__ - INFO - 🔍 1차 검색 모드: 0개의 문서 검색
2025-09-29 14:22:39,550 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:39,554 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.1
2025-09-29 14:22:39,555 - __main__ - INFO - 🔍 2차 검색 모드: 0개의 문서 검색
2025-09-29 14:22:39,907 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:39,911 - __main__ - INFO - 🔍 3차 검색 모드: 3개의 문서 검색


검색된 문서 수: 3
문서: 1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.


2025-09-29 14:22:40,621 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


문서 관련성: correct
문서: 1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.


2025-09-29 14:22:41,137 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


문서 관련성: correct
문서: 1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.


2025-09-29 14:22:41,718 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


문서 관련성: correct


`(2) Answer Generator`

In [11]:
def generator_answer(question, docs):
    template = """당신은 정확하고 도움되는 답변을 제공하는 AI 어시스턴트입니다.
    
        [지침]
        1. 제공된 문맥만을 사용하여 답변
        2. 불확실한 경우 명확히 표시
        3. 간결하되 완전한 답변 제공

        [문맥]
        {context}

        [질문]
        {question}

        [답변]"""

    prompt = ChatPromptTemplate.from_template(template)
    llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)    

    def format_docs(docs):
        return "\n\n".join([d.page_content for d in docs])
    
    rag_chain = prompt | llm | StrOutputParser()
    
    generation = rag_chain.invoke({"context": format_docs(docs), "question": question})

    return generation


# 검색된 문서를 기반으로 질문에 대한 답변 생성
generation = generator_answer(question, docs=retrieved_docs)
print(generation)

2025-09-29 14:22:45,740 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


네, 시그니처 메뉴로 21일간 건조 숙성한 최상급 한우 등심을 사용한 '시그니처 스테이크'가 있습니다. 가격은 ₩35,000이며, 로즈메리 감자와 그릴드 아스파라거스가 곁들여지고 레드와인 소스와 함께 제공됩니다.


`(3) Question Re-writer`

In [12]:
def rewrite_question(question: str) -> str:
    """
    주어진 질문을 벡터 저장소 검색에 최적화된 형태로 다시 작성합니다.
    """
    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

    system_prompt = """당신은 검색 최적화 전문가입니다.

    질문 개선 전략:
    1. 핵심 키워드 추출 및 강조
    2. 모호한 대명사를 구체적 용어로 대체
    3. 동의어 및 관련 용어 추가
    4. 시간적/공간적 맥락 명확화
    5. 복합 질문을 단순 질문으로 분해

    예시:
    - 원본: "그거 얼마야?"
    - 개선: "스테이크 메뉴 가격 정보"
    
    - 원본: "여기서 뭐가 제일 맛있어?"
    - 개선: "레스토랑 인기 메뉴 추천 시그니처 요리"

    주의사항:
    - 원래 의도 유지
    - 과도한 확장 지양
    - 검색 친화적 표현 사용"""

    re_write_prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", """[원본 질문]
{question}

검색에 최적화된 질문으로 재작성하세요. 
간결하고 명확하게, 핵심 키워드를 포함하여."""),
    ])

    question_rewriter = re_write_prompt | llm | StrOutputParser()
    rewritten_question = question_rewriter.invoke({"question": question})

    return rewritten_question

# 질문 다시 쓰기 테스트
rewritten_question = rewrite_question(question)
print(f"원본 질문: {question}")
print(f"다시 쓴 질문: {rewritten_question}")

2025-09-29 14:22:50,862 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


원본 질문: 시그니처 메뉴가 있나요?
다시 쓴 질문: 레스토랑 시그니처 메뉴 유무 확인


`(4) Knowledge Refiner`

In [13]:
class RefinedKnowledge(BaseModel):
    """
    문서에서 추출된 정제된 지식 조각을 나타냅니다.
    """
    knowledge_strip: str = Field(description="문서에서 추출된 정제된 지식 조각")
    binary_score: str = Field(
        description="문서가 질문과 관련이 있는지 여부, 'yes' 또는 'no'"
    )

structured_llm_refiner = llm.with_structured_output(RefinedKnowledge)

# 지식 정제를 위한 프롬프트
refine_system_prompt = """
당신은 지식 정제 전문가입니다. 주어진 질문과 관련하여 문서에서 핵심 정보를 추출하고 관련성을 평가하는 것이 당신의 임무입니다.

[지시사항]
1. 질문과 문서를 주의 깊게 읽으세요.
2. 질문에 답하는 데 관련이 있는 문서의 핵심 정보들을 식별하세요.
3. 각 핵심 정보에 대해:
   a. 간결하게 추출하고 요약하세요 (정보당 1-2문장을 목표로 함).
   b. 질문과의 관련성을 'yes' (관련 있음) 또는 'no' (관련 없음)로 평가하세요.
4. 각 정보를 다음 형식으로 새 줄에 제시하세요:
   [추출된 정보] (yes/no)

[예시 출력]
AI 시스템은 훈련 데이터에 존재하는 편향을 나타낼 수 있습니다. (yes)
의사결정에서 AI 사용은 개인정보 보호 우려를 제기합니다. (yes)
기계학습 모델은 상당한 컴퓨팅 자원을 필요로 합니다. (no)

[참고사항]
사실적이고 객관적인 정보 추출에 집중하세요. 개인적인 의견이나 추측은 피하세요. 3-5개의 핵심 정보 제공을 목표로 하되, 문서에 관련 내용이 특히 풍부한 경우 더 많이 포함해도 됩니다.
"""

refine_prompt = ChatPromptTemplate.from_messages([
    ("system", refine_system_prompt),
    ("human", "[문서]\n{document}\n\n[사용자 질문]\n{question}"),
])

# Knowledge Refiner 파이프라인 구성
knowledge_refiner = refine_prompt | structured_llm_refiner

# 지식 정제 실행
retrieved_docs = vector_db.multi_stage_search(question)
print(f"검색된 문서 수: {len(retrieved_docs)}")

for test_chunk in retrieved_docs:
    print("문서:", test_chunk.page_content)

    refined_knowledge = knowledge_refiner.invoke({"question": question, "document": test_chunk})
    print(f"정제된 지식: {refined_knowledge.knowledge_strip}")
    print(f"정제된 지식 평가: {refined_knowledge.binary_score}")
    print("=====================================")


2025-09-29 14:22:56,136 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:56,169 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.3
2025-09-29 14:22:56,171 - __main__ - INFO - 🔍 1차 검색 모드: 0개의 문서 검색
2025-09-29 14:22:56,683 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:56,686 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.1
2025-09-29 14:22:56,687 - __main__ - INFO - 🔍 2차 검색 모드: 0개의 문서 검색
2025-09-29 14:22:57,219 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:22:57,224 - __main__ - INFO - 🔍 3차 검색 모드: 3개의 문서 검색


검색된 문서 수: 3
문서: 1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.


2025-09-29 14:22:58,389 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


정제된 지식: 시그니처 스테이크는 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. (yes)
정제된 지식 평가: yes
문서: 1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.


2025-09-29 14:22:59,748 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


정제된 지식: 시그니처 스테이크는 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용하며 미디엄 레어로 조리됩니다. (yes)
정제된 지식 평가: yes
문서: 1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.


2025-09-29 14:23:00,986 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


정제된 지식: 시그니처 스테이크는 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용하며 미디엄 레어로 조리됩니다. (yes)
정제된 지식 평가: yes


### 3-3. LangGraph로 그래프 구현

`(1) 그래프 State 생성`

In [14]:
from typing import TypedDict, Union, List, Dict, Tuple, Any, Annotated
from langchain_core.documents import Document
import operator

class GraphState(TypedDict):
    """
    Corrective-RAG 그래프 상태
    """
    question: str
    generation: str
    retrieved_documents: List[Tuple[Document, str]]  # 검색 문서 리스트 (문서, 점수) -> 벡터 저장소 검색 결과 (내부지식)
    knowledge_strips: List[Tuple[Document, str]]     # 지식 보강한 결과 리스트 (문서, 점수) -> 최종 생성에 사용되는 문서 (내부지식 + 외부지식)
    num_generations: int
    # Send 결과를 수집하기 위한 필드 - reducer 사용
    graded_documents: Annotated[list, operator.add]   # Map-Reduce에서 평가 결과를 임시로 수집
    refined_knowledge: Annotated[list, operator.add]  # Map-Reduce 패턴에서 병렬 처리 결과를 임시로 수집하는 필드


`(2) Node 구성`

In [15]:
def retrieve(state: GraphState) -> GraphState:
    """문서를 검색하는 함수"""
    logging.info("--- 문서 검색 ---")
    question = state["question"]

    # 문서 검색 로직 (ambiguous 상태로 초기화)
    retrieved_documents = vector_db.multi_stage_search(question)
    retrieved_documents = [(doc, "ambiguous") for doc in retrieved_documents]
    return {"retrieved_documents": retrieved_documents}

def web_search(state: GraphState) -> GraphState:
    """웹 검색을 수행하는 함수"""
    logging.info("--- 웹 검색 ---")
    question = state["question"]
    
    # 웹 검색 로직 (ambiguous 상태로 초기화)
    search_results = search_tool.invoke(question)['results']
    retrieved_documents = [(Document(page_content=str(result)), "ambiguous") 
                          for result in search_results]
    return {"retrieved_documents": retrieved_documents}

In [16]:
# Map 단계: 문서 평가를 위한 Send 생성
def distribute_documents_for_grading(state: GraphState):
    """문서 평가를 위해 각 문서를 개별 노드로 보내는 함수"""
    retrieved_documents = state.get("retrieved_documents", [])
    question = state["question"]
    
    # Send 객체들의 리스트를 반환
    return [
        Send("grade_single_document", {
            "question": question, 
            "document": doc, 
            "grade": grade
        }) 
        for doc, grade in retrieved_documents
    ]

# Send를 위한 개별 문서 평가 함수
def grade_single_document(state: Dict) -> Dict:
    """개별 문서의 관련성을 평가하는 함수 (Send용)"""
    logging.info("--- 개별 문서 관련성 평가 ---")
    question = state["question"]
    document = state["document"]
    
    score = retrieval_grader.invoke({"question": question, "document": document.page_content})
    grade = score.relevance_score.lower()
    
    if grade == "correct":
        logging.info("---문서 관련성: 있음---")
        return {"graded_documents": [(document, "correct")]}
    elif grade == "incorrect":
        logging.info("---문서 관련성: 없음---")
        return {"graded_documents": [(document, "incorrect")]}
    else:
        logging.info("---문서 관련성: 모호함---")
        return {"graded_documents": [(document, "ambiguous")]}

In [17]:
# Map 단계: 지식 정제를 위한 Send 생성
def distribute_documents_for_refining(state: GraphState):
    """지식 정제를 위해 각 문서를 개별 노드로 보내는 함수"""
    graded_documents = state.get("graded_documents", [])
    question = state["question"]
    
    # graded_documents를 평면화
    flattened_docs = []
    for item in graded_documents:
        if isinstance(item, list):
            flattened_docs.extend(item)
        else:
            flattened_docs.append(item)
    
    # Send 객체들의 리스트를 반환
    return [
        Send("refine_single_knowledge", {
            "question": question, 
            "document": doc, 
            "grade": grade
        }) 
        for doc, grade in flattened_docs
    ]

# Send를 위한 개별 지식 정제 함수
def refine_single_knowledge(state: Dict) -> Dict:
    """개별 문서의 지식을 정제하는 함수 (Send용)"""
    logging.info("--- 개별 지식 정제 ---")
    question = state["question"]
    document = state["document"]
    grade = state.get("grade", "")
    
    # 관련성이 없는 문서는 제외
    if grade == "incorrect":
        return {"refined_knowledge": []}
    
    refined_knowledge = knowledge_refiner.invoke({"question": question, "document": document.page_content})
    knowledge = refined_knowledge.knowledge_strip
    binary_score = refined_knowledge.binary_score
    
    if binary_score == "yes":
        logging.info("---정제된 지식: 추가---")
        return {"refined_knowledge": [(Document(page_content=knowledge), "correct")]}
    else:
        logging.info("---정제된 지식: 제외---")
        return {"refined_knowledge": []}

In [18]:
# Reduce 단계: 평가된 문서들을 수집
def collect_graded_documents(state: GraphState) -> GraphState:
    """평가된 문서들을 수집하는 함수 (Reduce 단계)"""
    logging.info("--- 문서 평가 결과 수집 ---")
    graded_documents = state.get("graded_documents", [])
    
    logging.info(f"수집된 graded_documents: {len(graded_documents)}개")
    
    # 중첩된 리스트를 평면화
    flattened_docs = []
    for item in graded_documents:
        if isinstance(item, list):
            # 리스트인 경우 각 요소 추가
            for sub_item in item:
                if sub_item:  # 빈 요소가 아닌 경우만
                    flattened_docs.append(sub_item)
        elif item:  # 단일 요소이고 비어있지 않은 경우
            flattened_docs.append(item)
    
    logging.info(f"--- 총 {len(flattened_docs)}개 문서 평가 완료 ---")
    return {"retrieved_documents": flattened_docs, "graded_documents": []}  # 리셋


# Reduce 단계: 정제된 지식들을 수집
def collect_refined_knowledge(state: GraphState) -> GraphState:
    """정제된 지식들을 수집하는 함수 (Reduce 단계)"""
    logging.info("--- 정제된 지식 수집 ---")
    refined_knowledge = state.get("refined_knowledge", [])
    
    logging.info(f"수집된 refined_knowledge: {refined_knowledge}")
    
    # 중첩된 리스트를 평면화하고 빈 리스트 제거
    knowledge_strips = []
    for item in refined_knowledge:
        if isinstance(item, list):
            # 리스트인 경우 각 요소 확인
            for sub_item in item:
                if sub_item:  # 빈 요소가 아닌 경우만
                    knowledge_strips.append(sub_item)
        elif item:  # 단일 요소이고 비어있지 않은 경우
            knowledge_strips.append(item)
    
    logging.info(f"--- 총 {len(knowledge_strips)}개 지식 정제 완료 ---")
    return {"knowledge_strips": knowledge_strips, "refined_knowledge": []}  # 리셋

In [19]:
def generate(state: GraphState) -> GraphState:
    """답변을 생성하는 함수"""
    logging.info("--- 답변 생성 ---")
    num_generations = state.get("num_generations", 0)
    question = state["question"]
    knowledge_strips = state.get("knowledge_strips", [])
    
    # 지식이 없는 경우 처리
    if not knowledge_strips:
        generation = "죄송합니다. 질문에 대한 충분한 정보를 찾을 수 없습니다."
    else:
        # RAG를 이용한 답변 생성
        doc_texts = [doc for doc, _ in knowledge_strips]
        generation = generator_answer(question, docs=doc_texts)
    
    # 생성 횟수 업데이트
    num_generations += 1
    return {"generation": generation, "num_generations": num_generations}

def transform_query(state: GraphState) -> GraphState:
    """질문을 개선하는 함수"""
    logging.info("--- 질문 개선 ---")
    question = state["question"]
    # 질문 재작성
    rewritten_question = rewrite_question(question)
    return {"question": rewritten_question}

`(3) Edge 구성`

In [20]:
def decide_to_generate(state: GraphState) -> str:
    """답변 생성 여부를 결정하는 함수"""
    logging.info("--- 평가된 문서 분석 ---")
    knowledge_strips = state.get("knowledge_strips", [])
    num_generations = state.get("num_generations", 0)
    
    # 생성 횟수가 3회 이상이면 종료
    if num_generations >= 3:
        logging.info("--- 생성 횟수 초과: 종료 ---")
        return "generate"
    
    if not knowledge_strips:
        logging.info("--- 결정: 모든 문서가 질문과 관련이 없음, 질문 개선 필요 (-> transform_query)---")
        return "transform_query"
    else:
        logging.info("--- 결정: 답변 생성 (-> generate)---")
        return "generate"

`(4) 그래프 연결`

In [22]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# 워크플로우 그래프 초기화
builder = StateGraph(GraphState)

# 노드 정의
builder.add_node("retrieve", retrieve)  # 문서 검색
builder.add_node("web_search", web_search)  # 웹 검색

# Map-Reduce를 위한 노드들
builder.add_node("grade_single_document", grade_single_document)  # 개별 문서 평가 (Map)
builder.add_node("collect_graded_documents", collect_graded_documents)  # 문서 평가 결과 수집 (Reduce)
builder.add_node("refine_single_knowledge", refine_single_knowledge)  # 개별 지식 정제 (Map)
builder.add_node("collect_refined_knowledge", collect_refined_knowledge)  # 지식 정제 결과 수집 (Reduce)

# 기타 노드들
builder.add_node("generate", generate)  # 답변 생성
builder.add_node("transform_query", transform_query)  # 질문 개선

# 경로 정의
builder.add_edge(START, "retrieve")

# 문서 평가를 위한 Map-Reduce 패턴
# conditional_edges에서 Send를 반환하는 함수를 전달
builder.add_conditional_edges(
    "retrieve",
    distribute_documents_for_grading,  # Map: 각 문서를 개별 평가로 분산
    ["grade_single_document"]  # Send가 보낼 수 있는 노드 목록
)
builder.add_edge("grade_single_document", "collect_graded_documents")  # Reduce: 평가 결과 수집

# 지식 정제를 위한 Map-Reduce 패턴
builder.add_conditional_edges(
    "collect_graded_documents",
    distribute_documents_for_refining,  # Map: 각 문서를 개별 정제로 분산
    ["refine_single_knowledge"]  # Send가 보낼 수 있는 노드 목록
)
builder.add_edge("refine_single_knowledge", "collect_refined_knowledge")  # Reduce: 정제 결과 수집

# 조건부 엣지 추가: 문서 평가 후 결정
builder.add_conditional_edges(
    "collect_refined_knowledge",
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "generate": "generate",
    },
)

# 웹 검색 후 다시 문서 평가로 진입
builder.add_conditional_edges(
    "web_search",
    distribute_documents_for_grading,  # 웹 검색 후에도 Map-Reduce 패턴 적용
    ["grade_single_document"]
)

# 추가 경로
builder.add_edge("transform_query", "web_search")
builder.add_edge("generate", END)

# 그래프 컴파일
graph = builder.compile()

# 그래프 시각화
#display(Image(graph.get_graph().draw_mermaid_png()))

`(5) 그래프 실행`

In [23]:
# 첫 번째 예시
inputs = {"question": "스테이크 메뉴의 가격은 얼마인가요?"}
final_state = graph.invoke(inputs)

2025-09-29 14:24:01,732 - root - INFO - --- 문서 검색 ---
2025-09-29 14:24:02,117 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:24:02,121 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.3
2025-09-29 14:24:02,122 - __main__ - INFO - 🔍 1차 검색 모드: 0개의 문서 검색
2025-09-29 14:24:02,395 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:24:02,399 - __main__ - INFO - 🔍 2차 검색 모드: 5개의 문서 검색
2025-09-29 14:24:02,402 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:02,402 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:02,403 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:02,404 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:02,404 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:02,997 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-29 14:2

In [24]:
# 최종 답변
print(final_state.get("generation", "답변 생성 실패"))

샤토브리앙 스테이크는 ₩42,000이고, 시그니처 스테이크는 ₩35,000입니다.


In [25]:
# 두 번째 예시
inputs = {"question": "스테이크에 어울리는 와인을 추천해주세요."}
final_state = graph.invoke(inputs)

2025-09-29 14:24:08,433 - root - INFO - --- 문서 검색 ---
2025-09-29 14:24:09,035 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:24:09,042 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.3
2025-09-29 14:24:09,043 - __main__ - INFO - 🔍 1차 검색 모드: 0개의 문서 검색
2025-09-29 14:24:09,361 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:24:09,364 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.1
2025-09-29 14:24:09,365 - __main__ - INFO - 🔍 2차 검색 모드: 0개의 문서 검색
2025-09-29 14:24:09,671 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:24:09,675 - __main__ - INFO - 🔍 3차 검색 모드: 10개의 문서 검색
2025-09-29 14:24:09,680 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:09,682 - root - INFO - --- 개별 문서 관련성 평

NameError: name 'search_tool' is not defined

In [26]:
# 최종 답변
print(final_state.get("generation", "답변 생성 실패"))

샤토브리앙 스테이크는 ₩42,000이고, 시그니처 스테이크는 ₩35,000입니다.


In [27]:
# 세 번째 예시
inputs = {"question": "해산물 요리를 추천해주세요."}
final_state = graph.invoke(inputs)

2025-09-29 14:24:29,413 - root - INFO - --- 문서 검색 ---
2025-09-29 14:24:29,739 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:24:29,743 - langchain_core.vectorstores.base - WARNING - No relevant docs were retrieved using the relevance score threshold 0.3
2025-09-29 14:24:29,743 - __main__ - INFO - 🔍 1차 검색 모드: 0개의 문서 검색
2025-09-29 14:24:30,018 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 14:24:30,021 - __main__ - INFO - 🔍 2차 검색 모드: 5개의 문서 검색
2025-09-29 14:24:30,024 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:30,024 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:30,025 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:30,025 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:30,026 - root - INFO - --- 개별 문서 관련성 평가 ---
2025-09-29 14:24:30,542 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-29 14:2

In [28]:
# 최종 답변
print(final_state.get("generation", "답변 생성 실패"))

해산물 파스타와 씨푸드 빠에야를 추천합니다. 해산물 파스타는 링귀네 파스타에 새우, 홍합, 오징어 등 신선한 해산물과 토마토 소스, 마늘, 올리브 오일, 파슬리로 풍미를 더한 메뉴입니다. 씨푸드 빠에야는 스페인 전통 방식으로 조리한 홍합, 새우, 오징어 등 신선한 해산물이 듬뿍 들어간 요리입니다.


---
### **[실습]**

- State, Node, Edge를 직접 구성하여, Corrective RAG 시스템을 구현합니다. 
- 그래프 구현을 통해 RAG 품질을 개선하는 과정을 이해합니다.
- 데이터:
    - data/housing_leasing_law.pdf
    - data/labor_law.pdf
    - data/personal_info_law.pdf

In [41]:
from transformers import AutoTokenizer 

# Huggingface Tokenizer 초기화
bge_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")


from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화 (근로기준법 문서)
pdf_loader = PyPDFLoader('data/personal_info_law.pdf', mode='single')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 재귀적 텍스트 분할기 초기화 (토큰 수 기준 분할)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=bge_tokenizer,
    chunk_size=500, 
    chunk_overlap=100,
    separators=['\n\n', '\n', r'(?<=[.!?])\s+'],
)

# split_documents() 메서드 사용 : Document 객체를 여러 개의 작은 청크 문서로 분할
chunks = text_splitter.split_documents(pdf_docs) 

print(f"생성된 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")

# 각 청크의 시작 부분과 끝 부분 확인
for chunk in chunks[:5]:
    print(f"토큰 개수: {len(bge_tokenizer.encode(chunk.page_content))}")
    print(f"{chunk.page_content[:100]}...{chunk.page_content[-100:]}")
    print("="*100)


from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings   

# Ollama 임베딩 모델 생성
embeddings_ollama = OllamaEmbeddings(
    model="bge-m3"
)
embeddings_openai = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024  # 차원 설정
)

# Chroma 벡터 저장소 생성하기
chroma_db = Chroma.from_documents(  
    documents=chunks,
    embedding=embeddings_openai,    # 임베딩 사용
    collection_name="personal_info_law",    # 컬렉션 이름
    persist_directory="./chroma_db",
    collection_metadata = {'hnsw:space': 'cosine'}, # l2, ip, cosine 중에서 선택 
)

PDF 문서 개수: 1
생성된 청크 수: 97
각 청크의 길이: [1125, 1003, 1147, 1008, 1114, 903, 980, 859, 1091, 968, 1031, 1090, 1201, 1132, 1176, 1133, 1082, 1159, 1060, 1220, 984, 1129, 1029, 1148, 1148, 1033, 1085, 967, 1099, 1096, 891, 1114, 923, 1042, 1125, 1154, 1039, 1041, 1216, 1094, 1189, 1067, 1166, 1092, 1142, 1131, 1044, 1032, 1008, 1129, 1035, 1166, 1010, 1126, 1051, 946, 1052, 951, 1053, 959, 910, 1085, 962, 930, 1018, 849, 1022, 885, 878, 997, 976, 1162, 1082, 1221, 1056, 930, 1069, 902, 1095, 849, 978, 1104, 1021, 1009, 1024, 879, 966, 872, 859, 998, 884, 979, 900, 843, 918, 780, 653]
토큰 개수: 485
법제처                                                            1                                    ...하여 알아볼 수 있는 정보. 이 경우
쉽게 결합할 수 있는지 여부는 다른 정보의 입수 가능성 등 개인을 알아보는 데 소요되는 시간, 비용, 기술 등
을 합리적으로 고려하여야 한다.
토큰 개수: 495
가. 성명, 주민등록번호 및 영상 등을 통하여 개인을 알아볼 수 있는 정보
나. 해당 정보만으로는 특정 개인을 알아볼 수 없더라도 다른 정보와 쉽게 결합하여 알아볼 수 있는 정보....보처리기기”란 일정한 공간에 설치되어 지속적 또는 주기적으로 사람 또는 사물의 영상 등을 촬영
하거나 이를 유ㆍ무선망을 통하여 전송하는 장치로서 대통령령으로

2025-09-29 14:44:43,111 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [50]:
# 여기에 코드를 작성하세요.


from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

class AdaptiveVectorStore:
    """적응형 검색을 위한 벡터 저장소"""
    
    def __init__(self, collection_name: str, persist_dir: str = "./chroma_db"):
        # 임베딩 모델 사용 
        self.embeddings = OpenAIEmbeddings(
            model="text-embedding-3-small",
            dimensions=1024  # 차원 설정
        )
        
        # Chroma DB 초기화
        self.vector_db = Chroma(
            embedding_function=self.embeddings,
            collection_name=collection_name,
            persist_directory=persist_dir,
        )
        
        # 다단계 검색 전략
        self.search_configs = {
            "initial": {"k": 3, "score_threshold": 0.3},
            "expanded": {"k": 5, "score_threshold": 0.1},
            "exhaustive": {"k": 10, "score_threshold": 0.0}
        }

        logger.info(f"✅ Vector store '{collection_name}' initialized")

    def multi_stage_search(self, query: str):
        """단계별 검색 전략"""
        # 1차: 정밀 검색
        initial = self.vector_db.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs=self.search_configs["initial"]
        ).invoke(query)

        logger.info(f"🔍 1차 검색 모드: {len(initial)}개의 문서 검색")

        if len(initial) < 1:
            # 2차: 확장 검색
            expanded = self.vector_db.as_retriever(
                search_type="similarity_score_threshold",
                search_kwargs=self.search_configs["expanded"]
            ).invoke(query)

            logger.info(f"🔍 2차 검색 모드: {len(expanded)}개의 문서 검색")

            if len(expanded) < 1:
                # 3차: 포괄 검색
                exhaustive = self.vector_db.as_retriever(
                    search_type="similarity_score_threshold",
                    search_kwargs=self.search_configs["exhaustive"]
                ).invoke(query)

                logger.info(f"🔍 3차 검색 모드: {len(exhaustive)}개의 문서 검색")

                return exhaustive
            
            return expanded
        else:
            return initial


# 사용 예시
#vector_db = AdaptiveVectorStore(collection_name="housing_leasing_law")
#results = vector_db.multi_stage_search("손해배상을 청구할 경우에는 노동위원회에 신청할 수 있는가?")
#results = vector_db.multi_stage_search("임차인이 상속인 없이 사망한 경우에는 누가 임차인의 권리와 의무를 승계할 수 있는가?")

init_question = "임차인이 상속인 없이 사망한 경우에는 누가 임차인의 권리와 의무를 승계할 수 있는가?"

from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 문서 관련성 평가 결과를 위한 데이터 모델 정의
class GradeDocuments(BaseModel):
    """Three-class score for relevance check on retrieved documents."""
    relevance_score: Literal["correct", "incorrect", "ambiguous"] = Field(
        description="Document relevance to the question: 'correct', 'incorrect', or 'ambiguous'"
    )

# LLM 모델 초기화 및 구조화된 출력 설정
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# 문서 관련성 평가를 위한 시스템 프롬프트 정의
system_prompt = """
You are an expert evaluator tasked with assessing the relevance of retrieved documents to a user's question. Your role is crucial in enhancing the quality of information retrieval systems.

[평가 기준]
1. 키워드 관련성: 문서가 질문의 주요 단어나 유사어를 포함하는지 확인
2. 의미적 관련성: 문서의 전반적인 주제가 질문의 의도와 일치하는지 평가
3. 부분 관련성: 질문의 일부를 다루거나 맥락 정보를 제공하는 문서도 고려
4. 답변 가능성: 직접적인 답이 아니더라도 답변 형성에 도움될 정보 포함 여부 평가

[점수 체계]
- 'correct': 문서가 명확히 관련 있고, 질문에 답하는 데 필요한 정보를 포함함.
- 'incorrect': 문서가 명확히 무관하거나, 질문에 도움이 되지 않는 정보를 포함함.
- 'ambiguous': 문서의 관련성이 불분명하거나, 일부 관련 정보는 있지만 유용성이 확실하지 않음, 혹은 질문과 약간만 관련 있음.

[주의사항]
- 단순 단어 매칭이 아닌 질문의 전체 맥락을 고려하세요
- 완벽한 답변이 아니어도 유용한 정보가 있다면 관련 있다고 판단하세요

Your evaluation plays a critical role in improving the overall performance of the information retrieval system. Strive for balanced and thoughtful assessments.
"""

# 채점 프롬프트 템플릿 생성
grade_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Document: \n\n {document} \n\n Question: {question}"),
])

# Retrieval Grader 파이프라인 구성
retrieval_grader = grade_prompt | structured_llm_grader


# 관련성 평가 실행
#question = init_question
#retrieved_docs = vector_db.multi_stage_search(question)
#print(f"검색된 문서 수: {len(retrieved_docs)}")

#for test_chunk in retrieved_docs:
#    print("문서:", test_chunk.page_content)

#    relevance = retrieval_grader.invoke({"question": question, "document": test_chunk.page_content})
#    print(f"문서 관련성: {relevance.relevance_score}")
#    print("=====================================")


#question = init_question
#retrieved_docs = vector_db.multi_stage_search(question)
#print(f"검색된 문서 수: {len(retrieved_docs)}")

#for test_chunk in retrieved_docs:
#    print("문서:", test_chunk.page_content)

#    relevance = retrieval_grader.invoke({"question": question, "document": test_chunk.page_content})
#    print(f"문서 관련성: {relevance.relevance_score}")
#    print("=====================================")

def generator_answer(question, docs):
    template = """당신은 정확하고 도움되는 답변을 제공하는 AI 어시스턴트입니다.
    
        [지침]
        1. 제공된 문맥만을 사용하여 답변
        2. 불확실한 경우 명확히 표시
        3. 간결하되 완전한 답변 제공

        [문맥]
        {context}

        [질문]
        {question}

        [답변]"""

    prompt = ChatPromptTemplate.from_template(template)
    llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)    

    def format_docs(docs):
        return "\n\n".join([d.page_content for d in docs])
    
    rag_chain = prompt | llm | StrOutputParser()
    
    generation = rag_chain.invoke({"context": format_docs(docs), "question": question})

    return generation


# 검색된 문서를 기반으로 질문에 대한 답변 생성
#generation = generator_answer(question, docs=retrieved_docs)
#print(generation)

def rewrite_question(question: str) -> str:
    """
    주어진 질문을 벡터 저장소 검색에 최적화된 형태로 다시 작성합니다.
    """
    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

    system_prompt = """당신은 검색 최적화 전문가입니다.

    질문 개선 전략:
    1. 핵심 키워드 추출 및 강조
    2. 모호한 대명사를 구체적 용어로 대체
    3. 동의어 및 관련 용어 추가
    4. 시간적/공간적 맥락 명확화
    5. 복합 질문을 단순 질문으로 분해

    예시:
    - 원본: "그거 얼마야?"
    - 개선: "스테이크 메뉴 가격 정보"
    
    - 원본: "여기서 뭐가 제일 맛있어?"
    - 개선: "레스토랑 인기 메뉴 추천 시그니처 요리"

    주의사항:
    - 원래 의도 유지
    - 과도한 확장 지양
    - 검색 친화적 표현 사용"""

    re_write_prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", """[원본 질문]
{question}

검색에 최적화된 질문으로 재작성하세요. 
간결하고 명확하게, 핵심 키워드를 포함하여."""),
    ])

    question_rewriter = re_write_prompt | llm | StrOutputParser()
    rewritten_question = question_rewriter.invoke({"question": question})

    return rewritten_question

# 질문 다시 쓰기 테스트
#rewritten_question = rewrite_question(question)
#print(f"원본 질문: {question}")
#print(f"다시 쓴 질문: {rewritten_question}")

class RefinedKnowledge(BaseModel):
    """
    문서에서 추출된 정제된 지식 조각을 나타냅니다.
    """
    knowledge_strip: str = Field(description="문서에서 추출된 정제된 지식 조각")
    binary_score: str = Field(
        description="문서가 질문과 관련이 있는지 여부, 'yes' 또는 'no'"
    )

structured_llm_refiner = llm.with_structured_output(RefinedKnowledge)

# 지식 정제를 위한 프롬프트
refine_system_prompt = """
당신은 지식 정제 전문가입니다. 주어진 질문과 관련하여 문서에서 핵심 정보를 추출하고 관련성을 평가하는 것이 당신의 임무입니다.

[지시사항]
1. 질문과 문서를 주의 깊게 읽으세요.
2. 질문에 답하는 데 관련이 있는 문서의 핵심 정보들을 식별하세요.
3. 각 핵심 정보에 대해:
   a. 간결하게 추출하고 요약하세요 (정보당 1-2문장을 목표로 함).
   b. 질문과의 관련성을 'yes' (관련 있음) 또는 'no' (관련 없음)로 평가하세요.
4. 각 정보를 다음 형식으로 새 줄에 제시하세요:
   [추출된 정보] (yes/no)

[예시 출력]
AI 시스템은 훈련 데이터에 존재하는 편향을 나타낼 수 있습니다. (yes)
의사결정에서 AI 사용은 개인정보 보호 우려를 제기합니다. (yes)
기계학습 모델은 상당한 컴퓨팅 자원을 필요로 합니다. (no)

[참고사항]
사실적이고 객관적인 정보 추출에 집중하세요. 개인적인 의견이나 추측은 피하세요. 3-5개의 핵심 정보 제공을 목표로 하되, 문서에 관련 내용이 특히 풍부한 경우 더 많이 포함해도 됩니다.
"""

refine_prompt = ChatPromptTemplate.from_messages([
    ("system", refine_system_prompt),
    ("human", "[문서]\n{document}\n\n[사용자 질문]\n{question}"),
])

# Knowledge Refiner 파이프라인 구성
knowledge_refiner = refine_prompt | structured_llm_refiner

# 지식 정제 실행
retrieved_docs = vector_db.multi_stage_search(question)
print(f"검색된 문서 수: {len(retrieved_docs)}")

for test_chunk in retrieved_docs:
    print("문서:", test_chunk.page_content)

    refined_knowledge = knowledge_refiner.invoke({"question": question, "document": test_chunk})
    print(f"정제된 지식: {refined_knowledge.knowledge_strip}")
    print(f"정제된 지식 평가: {refined_knowledge.binary_score}")
    print("=====================================")




2025-09-29 15:04:42,600 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-29 15:04:42,783 - __main__ - INFO - 🔍 1차 검색 모드: 2개의 문서 검색


검색된 문서 수: 2
문서: ⑨ 금융기관등은 우선변제권을 행사하기 위하여 임차인을 대리하거나 대위하여 임대차를 해지할 수 없다.<신설
2013. 8. 13.>
[전문개정 2008. 3. 21.]
 
제3조의3(임차권등기명령) ① 임대차가 끝난 후 보증금이 반환되지 아니한 경우 임차인은 임차주택의 소재지를 관할하
는 지방법원ㆍ지방법원지원 또는 시ㆍ군 법원에 임차권등기명령을 신청할 수 있다. <개정 2013. 8. 13.>
② 임차권등기명령의 신청서에는 다음 각 호의 사항을 적어야 하며, 신청의 이유와 임차권등기의 원인이 된 사실을
소명(疎明)하여야 한다.<개정 2013. 8. 13.>
1. 신청의 취지 및 이유
2. 임대차의 목적인 주택(임대차의 목적이 주택의 일부분인 경우에는 해당 부분의 도면을 첨부한다)
3. 임차권등기의 원인이 된 사실(임차인이 제3조제1항ㆍ제2항 또는 제3항에 따른 대항력을 취득하였거나 제3조의
2제2항에 따른 우선변제권을 취득한 경우에는 그 사실)
4. 그 밖에 대법원규칙으로 정하는 사항
③ 다음 각 호의 사항 등에 관하여는 「민사집행법」 제280조제1항, 제281조, 제283조, 제285조, 제286조, 제288조제
1항, 같은 조 제2항 본문, 제289조, 제290조제2항 중 제288조제1항에 대한 부분, 제291조, 제292조제3항 및 제
293조를 준용한다. 이 경우 “가압류”는 “임차권등기”로, “채권자”는 “임차인”으로, “채무자”는 “임대인”으로 본다.<개
정 2023. 4. 18.>
1. 임차권등기명령의 신청에 대한 재판
2. 임차권등기명령의 결정에 대한 임대인의 이의신청 및 그에 대한 재판
3. 임차권등기명령의 취소신청 및 그에 대한 재판
4. 임차권등기명령의 집행


2025-09-29 15:04:44,940 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


정제된 지식: 문서에는 임차인이 상속인 없이 사망한 경우 임차인의 권리와 의무를 누가 승계하는지에 대한 내용이 포함되어 있지 않다. (no)
정제된 지식 평가: no
문서: 대법원규칙으로 정한다.<개정 2011. 4. 12.>
⑧ 임차인은 제1항에 따른 임차권등기명령의 신청과 그에 따른 임차권등기와 관련하여 든 비용을 임대인에게 청구
할 수 있다.
⑨ 금융기관등은 임차인을 대위하여 제1항의 임차권등기명령을 신청할 수 있다. 이 경우 제3항ㆍ제4항 및 제8항의
“임차인”은 “금융기관등”으로 본다.<신설 2013. 8. 13.>
[전문개정 2008. 3. 21.]
 
제3조의4(「민법」에 따른 주택임대차등기의 효력 등) ① 「민법」 제621조에 따른 주택임대차등기의 효력에 관하여는 제
3조의3제5항 및 제6항을 준용한다.
② 임차인이 대항력이나 우선변제권을 갖추고 「민법」 제621조제1항에 따라 임대인의 협력을 얻어 임대차등기를
신청하는 경우에는 신청서에 「부동산등기법」 제74조제1호부터 제6호까지의 사항 외에 다음 각 호의 사항을 적어
야 하며, 이를 증명할 수 있는 서면(임대차의 목적이 주택의 일부분인 경우에는 해당 부분의 도면을 포함한다)을 첨
부하여야 한다.<개정 2011. 4. 12., 2020. 2. 4.>
1. 주민등록을 마친 날
2. 임차주택을 점유(占有)한 날
3. 임대차계약증서상의 확정일자를 받은 날
[전문개정 2008. 3. 21.]
 
제3조의5(경매에 의한 임차권의 소멸) 임차권은 임차주택에 대하여 「민사집행법」에 따른 경매가 행하여진 경우에는 그
임차주택의 경락(競落)에 따라 소멸한다. 다만, 보증금이 모두 변제되지 아니한, 대항력이 있는 임차권은 그러하지 아
니하다.
[전문개정 2008. 3. 21.]


2025-09-29 15:04:46,476 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


정제된 지식: 임차인은 임차권등기명령 신청과 관련 비용을 임대인에게 청구할 수 있으며, 금융기관 등이 임차인을 대위하여 임차권등기명령을 신청할 수 있다. (no)
정제된 지식 평가: no
